In [ ]:
import tensorflow as tf
import torch
import numpy as np
import pandas as pd
import transformers

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Transformers version:", transformers.__version__)

print("\nTensorFlow GPUs:", tf.config.list_physical_devices("GPU"))
print("PyTorch CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import pandas as pd

file_path = "/kaggle/input/datasets/debarshichanda/goemotions/data/full_dataset/goemotions_1.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
# Check missing values and duplicate texts

print("Missing text values:", df["text"].isnull().sum())
print("Duplicate texts:", df["text"].duplicated().sum())

# Remove missing and duplicate text rows
df = df.dropna(subset=["text"])
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("\nDataset shape after cleaning:", df.shape)

# Display sample texts
df[["text"]].head()

In [ ]:
# Select emotion label columns

emotion_columns = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval',
    'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
    'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
    'pride', 'realization', 'relief', 'remorse', 'sadness',
    'surprise', 'neutral'
]

# Check emotion label distribution
emotion_counts = df[emotion_columns].sum().sort_values(ascending=False)

print("Number of emotion labels:", len(emotion_columns))
print("\nEmotion distribution:")
print(emotion_counts)

In [ ]:
from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Test tokenizer on one sample
sample_text = df["text"].iloc[0]

encoded_sample = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

print("Original text:")
print(sample_text)

print("\nInput IDs shape:", encoded_sample["input_ids"].shape)
print("Attention mask shape:", encoded_sample["attention_mask"].shape)

print("\nTokenization successful!")

In [ ]:
from sklearn.model_selection import train_test_split

# Features and multi-label targets
X = df["text"]
y = df[emotion_columns]

# 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Split temporary data into 10% validation and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

print("\nTotal samples:", len(X_train) + len(X_val) + len(X_test))

In [ ]:
# Tokenize train, validation and test data

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Training data tokenized:", len(train_encodings["input_ids"]))
print("Validation data tokenized:", len(val_encodings["input_ids"]))
print("Test data tokenized:", len(test_encodings["input_ids"]))

print("\nFull dataset tokenization successful!")

In [ ]:
import torch
from torch.utils.data import Dataset

class GoEmotionsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx].values,
            dtype=torch.float
        )

        return item

    def __len__(self):
        return len(self.labels)

train_dataset = GoEmotionsDataset(train_encodings, y_train)
val_dataset = GoEmotionsDataset(val_encodings, y_val)
test_dataset = GoEmotionsDataset(test_encodings, y_test)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

sample = train_dataset[0]

print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Labels shape:", sample["labels"].shape)

print("PyTorch datasets created successfully!")

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

num_labels = 28

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully!")
print("Device:", device)
print("Number of labels:", model.config.num_labels)
print("Problem type:", model.config.problem_type)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Labels:", batch["labels"].shape)

print("DataLoaders created successfully!")

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

epochs = 3

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("Optimizer created successfully!")
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Epochs:", epochs)
print("Total training steps:", total_steps)
print("Scheduler created successfully!")

In [ ]:
from tqdm.auto import tqdm
import torch

for epoch in range(epochs):

    print(f"\n===== Epoch {epoch + 1}/{epochs} =====")

    model.train()
    total_train_loss = 0

    progress_bar = tqdm(train_loader, desc="Training")

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_train_loss = total_train_loss / len(train_loader)

    print(
        f"Average training loss: "
        f"{average_train_loss:.4f}"
    )

print("\nTraining completed successfully!")

In [ ]:
torch.save(model.state_dict(), "emotion_model.pth")

print("Model saved successfully!")

In [ ]:
model.eval()

images, labels = next(iter(train_loader))

image = images[0].unsqueeze(0).to(device)
actual_label = labels[0].item()

with torch.no_grad():
    output = model(image)
    predicted_label = torch.argmax(output, dim=1).item()

print("Actual label:", actual_label)
print("Predicted label:", predicted_label)

In [ ]:
batch = next(iter(train_loader))

print("Number of values in batch:", len(batch))

for i, value in enumerate(batch):
    print(i, type(value))

In [ ]:
print("Batch contents:")

for i, value in enumerate(batch):
    print(i, value)

In [ ]:
model.eval()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    predictions = torch.argmax(outputs.logits, dim=1)

print("Actual labels:", labels[0])
print("Predicted label:", predictions[0].item())

In [ ]:
actual_label = torch.argmax(labels[0]).item()
predicted_label = predictions[0].item()

print("Actual label:", actual_label)
print("Predicted label:", predicted_label)

In [ ]:
print(model.config.id2label)

In [ ]:
print(train_dataset.features)

In [ ]:
emotion_labels = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

actual_emotion = emotion_labels[actual_label]
predicted_emotion = emotion_labels[predicted_label]

print("Actual Emotion:", actual_emotion)
print("Predicted Emotion:", predicted_emotion)

In [ ]:
text = "I am very happy today!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
).to(device)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)
    predicted_label = torch.argmax(outputs.logits, dim=1).item()

predicted_emotion = emotion_labels[predicted_label]

print("Input Text:", text)
print("Predicted Emotion:", predicted_emotion)

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)
        actual_labels = torch.argmax(labels, dim=1)

        correct += (predictions == actual_labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Model Accuracy: {accuracy:.2f}%")

In [ ]:
print("test_loader exists:", "test_loader" in globals())
print("val_loader exists:", "val_loader" in globals())

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)
        actual_labels = torch.argmax(labels, dim=1)

        correct += (predictions == actual_labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")